# ReciCall Priority Queue Evaluation

This notebook evaluates the generated ReciCall priority-queue results using Python's standard library. Run `npm.cmd run evaluate:algorithm` before opening it.

In [ ]:
from pathlib import Path
from collections import Counter
import csv

ROOT = Path.cwd()
if not (ROOT / 'evaluation').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'evaluation' / 'data'
RESULTS_DIR = ROOT / 'evaluation' / 'results'

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

students = read_csv(DATA_DIR / 'students.csv')
events = read_csv(DATA_DIR / 'participation_events.csv')
selection = read_csv(RESULTS_DIR / 'selection_results.csv')
replay = read_csv(RESULTS_DIR / 'replay_results.csv')
print(f'Project root: {ROOT}')
print(f'Students: {len(students)} | Evaluation sessions: {len(selection)}')

## Accuracy

Top-1 checks the first queue selection. Top-3 checks whether the teacher-labeled student appears in the first three positions.

In [ ]:
total = len(selection)
top1 = sum(row['top_1_correct'] == 'YES' for row in selection)
top3 = sum(row['top_3_correct'] == 'YES' for row in selection)
print(f'Top-1 accuracy: {top1}/{total} = {top1 / total * 100:.2f}%')
print(f'Top-3 accuracy: {top3}/{total} = {top3 / total * 100:.2f}%')

## Fairness and coverage

The fairness calculation matches ReciCall's Gini-based score: `(1 - Gini coefficient) x 100`.

In [ ]:
def fairness_score(counts):
    counts = list(counts)
    if not counts or sum(counts) == 0:
        return 100
    mean = sum(counts) / len(counts)
    difference = sum(abs(left - right) for left in counts for right in counts)
    gini = difference / (2 * len(counts) ** 2 * mean)
    return round(max(0, min(100, (1 - gini) * 100)))

student_ids = [student['student_id'] for student in students]
student_names = {student['student_id']: student['student_name'] for student in students}
historical_counter = Counter(row['student_id'] for row in events if row['event_type'] == 'participation')
replay_counter = Counter(row['selected_student_id'] for row in replay)
historical_counts = [historical_counter.get(student_id, 0) for student_id in student_ids]
replay_counts = [replay_counter.get(student_id, 0) for student_id in student_ids]

for label, counts in [('Historical records', historical_counts), ('Algorithm replay', replay_counts)]:
    coverage = sum(count > 0 for count in counts) / len(counts) * 100
    print(f'{label}: fairness={fairness_score(counts)}/100, gap={max(counts) - min(counts)}, coverage={coverage:.2f}%')

print('Student turn counts after algorithm replay:')
for student_id, count in zip(student_ids, replay_counts):
    print(f'  {student_names[student_id]}: {count}')

## Waiting time and formula checks

The evaluator reports true session waiting time separately from the algorithm's internal waiting value. A large difference indicates that the waiting-factor implementation needs review.

In [ ]:
true_wait = [float(row['true_wait_sessions']) for row in selection]
reported_wait = [float(row['algorithm_wait_value']) for row in selection]
formula_errors = sum(int(row['formula_errors_in_queue']) for row in selection)
capped_scores = sum(int(row['capped_scores_in_queue']) for row in selection)
print(f'Average true wait: {sum(true_wait) / len(true_wait):.2f} sessions')
print(f'Average algorithm-reported wait: {sum(reported_wait) / len(reported_wait):.2f}')
print(f'Formula errors: {formula_errors}')
print(f'Capped candidate scores: {capped_scores}')
print('Formula errors should be 0. Capped scores are expected but can create ties.')

## Interpretation

The included labels and events are synthetic. They verify the evaluation process, not real classroom performance. Replace `teacher_labels.csv` with labels collected from actual teacher reviews before using the results in the thesis. Report accuracy and fairness together.